# Data Cleaning and Temporal harmonization

## Load dataset

In [1]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User and Filter Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

In [5]:
# define initial variables
dataset_path = Path('../dataset/raw/gme/load/')
YEARS = range(2015, 2027)

# create prices dataframe
load_dfs = [
    pd.read_excel(
        dataset_path / f'GME-load-{year}.xlsx', 
        decimal=','
    )
    for year in YEARS
]

load_data = pd.concat(load_dfs, ignore_index=True)

In [6]:
load_data

,Data,Ora,Centro Nord,Centro Sud,Nord,Sardegna,Sicilia,Sud,Totale Italia,Calabria
0,01/01/2015,1,"2.670,851","4.590,186","10.774,037",997.138,"1.930,829","2.347,270","23.310,311",NaN
1,01/01/2015,2,"2.513,713","4.342,739","10.576,993",951.110,"1.821,231","2.213,368","22.419,154",NaN
2,01/01/2015,3,"2.392,789","4.028,419","10.169,339",903.001,"1.769,136","2.031,384","21.294,068",NaN
3,01/01/2015,4,"2.259,362","3.732,403","9.769,460",855.196,"1.673,111","1.869,842","20.159,374",NaN
4,01/01/2015,5,"2.245,990","3.363,157","9.613,212",855.747,"1.536,739","1.761,414","19.376,259",NaN
...,...,...,...,...,...,...,...,...,...,...
100770,30/06/2026,20,"4.370,9","8.561,8","27.660,0","1.422,8","2.603,7","2.926,7","48.437,3",891.5
100771,30/06/2026,21,"4.242,0","8.454,6","26.765,1","1.450,4","2.641,6","2.921,0","47.383,5",908.9
100772,30/06/2026,22,"4.090,8","8.224,8","25.945,6","1.419,2","2.566,3","2.825,6","45.951,1",878.8
100773,30/06/2026,23,"3.776,1","7.714,4","24.390,6","1.346,6","2.412,6","2.743,6","43.204,2",820.4


In [7]:
load_data.shape

(100775, 10)

In [8]:
# check what to clean
load_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100775 entries, 0 to 100774
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Data           100775 non-null  str   
 1   Ora            100775 non-null  int64 
 2   Centro Nord    100775 non-null  str   
 3   Centro Sud     100775 non-null  str   
 4   Nord           100775 non-null  str   
 5   Sardegna       100775 non-null  str   
 6   Sicilia        100775 non-null  str   
 7   Sud            100775 non-null  str   
 8   Totale Italia  100775 non-null  str   
 9   Calabria       48167 non-null   object
dtypes: int64(1), object(1), str(8)
memory usage: 7.7+ MB


In [9]:
load_data['Data'].describe()

count         100775
unique          4199
top       25/10/2015
freq              25
Name: Data, dtype: object

In [10]:
load_data['Ora'].describe()

count    100775.000000
mean         12.499995
std           6.922350
min           1.000000
25%           6.500000
50%          12.000000
75%          18.000000
max          25.000000
Name: Ora, dtype: float64

In [11]:
load_data[load_data.columns[2:]].describe()

,Centro Nord,Centro Sud,Nord,Sardegna,Sicilia,Sud,Totale Italia,Calabria
count,100775,100775,100775,100775,100775,100775,100775,48167
unique,96252,98012,99888,86668,93029,94112,100249,40696
top,"2.591,7","5.359,6","15.965,0",885.2,"1.780,8","1.570,2","30.693,9",576.8
freq,5,5,3,12,8,7,3,10


In [12]:
# days with more than 24 hourly periods
daily_periods = (
    load_data
    .groupby('Data')['Ora']
    .agg(['count', 'min', 'max'])
)

daily_periods[daily_periods['count'] != 24]

,count,min,max
Data,,,
25/03/2018,23,1,23
25/10/2015,25,1,25
25/10/2020,25,1,25
26/03/2017,23,1,23
26/03/2023,23,1,23
26/10/2025,25,1,25
27/03/2016,23,1,23
27/03/2022,23,1,23
27/10/2019,25,1,25


In [13]:
daily_periods['count'].value_counts().sort_index()

count
23      12
24    4176
25      11
Name: count, dtype: int64

In [15]:
load_data[
    load_data['Data'].isin(
        daily_periods.query('count != 24').index
    )
][['Data', 'Ora', 'Totale Italia']]

,Data,Ora,Totale Italia
2088,29/03/2015,1,"24.120,304"
2089,29/03/2015,2,"22.191,033"
2090,29/03/2015,3,"21.105,774"
2091,29/03/2015,4,"21.305,470"
2092,29/03/2015,5,"21.346,699"
...,...,...,...
98538,29/03/2026,19,"31.063,3"
98539,29/03/2026,20,"31.372,3"
98540,29/03/2026,21,"29.773,3"
98541,29/03/2026,22,"27.653,0"


In [17]:
# 1. convert date
load_data['Data'] = pd.to_datetime(
    load_data['Data'],
    dayfirst=True,
    errors='coerce'
)

# 2. inspect the number of periods per day
daily_periods = (
    load_data
    .groupby('Data')['Ora']
    .agg(['count', 'min', 'max'])
)

print(daily_periods['count'].value_counts().sort_index())

# 3. Inspect non-standard days
print(
    load_data[
        load_data['Data'].isin(
            daily_periods.query('count != 24').index
        )
    ][['Data', 'Ora', 'Totale Italia']]
)

count
23      12
24    4176
25      11
Name: count, dtype: int64
            Data  Ora Totale Italia
2088  2015-03-29    1    24.120,304
2089  2015-03-29    2    22.191,033
2090  2015-03-29    3    21.105,774
2091  2015-03-29    4    21.305,470
2092  2015-03-29    5    21.346,699
...          ...  ...           ...
98538 2026-03-29   19      31.063,3
98539 2026-03-29   20      31.372,3
98540 2026-03-29   21      29.773,3
98541 2026-03-29   22      27.653,0
98542 2026-03-29   23      25.310,8

[551 rows x 3 columns]


In [20]:
# spring DST transition
spring_dates = daily_periods.query('count == 23').index

for date in spring_dates:
    print(f'\n--- {date} ---')
    print(
        load_data.loc[
            load_data['Data'] == date,
            ['Data', 'Ora', 'Totale Italia' ]
        ].to_string(index=False)
    )


--- 2015-03-29 00:00:00 ---
      Data  Ora Totale Italia
2015-03-29    1    24.120,304
2015-03-29    2    22.191,033
2015-03-29    3    21.105,774
2015-03-29    4    21.305,470
2015-03-29    5    21.346,699
2015-03-29    6    21.569,261
2015-03-29    7    22.347,366
2015-03-29    8    24.148,307
2015-03-29    9    25.622,057
2015-03-29   10    26.204,025
2015-03-29   11    25.885,065
2015-03-29   12    26.025,146
2015-03-29   13    24.540,336
2015-03-29   14    23.984,348
2015-03-29   15    23.559,643
2015-03-29   16    23.620,152
2015-03-29   17    25.344,926
2015-03-29   18    27.923,163
2015-03-29   19    31.045,215
2015-03-29   20    31.692,215
2015-03-29   21    30.179,584
2015-03-29   22    27.761,037
2015-03-29   23    25.024,539

--- 2016-03-27 00:00:00 ---
      Data  Ora Totale Italia
2016-03-27    1    22.844,756
2016-03-27    2    20.825,828
2016-03-27    3    19.397,436
2016-03-27    4    19.331,908
2016-03-27    5    19.414,164
2016-03-27    6    19.430,992
2016-03-27  

In [21]:
# autumn DST transition
autumn_dates = daily_periods.query('count == 25').index

for date in autumn_dates:
    print(f'\n--- {date} ---')
    print(
        load_data.loc[
            load_data['Data'] == date,
            ['Data', 'Ora', 'Totale Italia' ]
        ].to_string(index=False)
    )


--- 2015-10-25 00:00:00 ---
      Data  Ora Totale Italia
2015-10-25    1    23.852,901
2015-10-25    2    22.218,764
2015-10-25    3    21.386,027
2015-10-25    4    21.554,636
2015-10-25    5    20.973,890
2015-10-25    6    21.014,990
2015-10-25    7    21.935,183
2015-10-25    8    22.801,795
2015-10-25    9    23.609,703
2015-10-25   10    25.012,742
2015-10-25   11    25.824,848
2015-10-25   12    26.884,624
2015-10-25   13    26.322,012
2015-10-25   14    25.995,502
2015-10-25   15    24.602,485
2015-10-25   16    24.212,730
2015-10-25   17    24.051,622
2015-10-25   18    25.087,727
2015-10-25   19    27.783,242
2015-10-25   20    30.838,048
2015-10-25   21    32.384,403
2015-10-25   22    31.141,812
2015-10-25   23    29.482,083
2015-10-25   24    27.372,631
2015-10-25   25    24.897,960

--- 2016-10-30 00:00:00 ---
      Data  Ora Totale Italia
2016-10-30    1    23.979,491
2016-10-30    2    22.687,338
2016-10-30    3    21.430,453
2016-10-30    4    21.167,967
2016-10-30  

In [22]:
load_data['datetime'] = (
    load_data['Data']
    + pd.to_timedelta(load_data['Ora'] - 1, unit='h')
)

diffs = load_data['datetime'].diff()

diffs.value_counts().sort_index()

datetime
0 days 00:00:00        11
0 days 01:00:00    100751
0 days 02:00:00        12
Name: count, dtype: int64

Source for period meaning for GME: https://gme.mercatoelettrico.org/Portals/0/Documents/it-IT/20250101DTF01rev3MPE.pdf

In [23]:
# make a copy
load_data = load_data.copy()

# convert Date
load_data['Date'] = pd.to_datetime(
    load_data['Data'],
    dayfirst=True,
    errors='coerce'
)

# make sure Ora is numeric
load_data['Ora'] = pd.to_numeric(
    load_data['Ora'],
    errors='coerce'
)

In [24]:
periods_per_day = (
    load_data
    .groupby('Date')['Ora']
    .transform('count')
)

load_data['periods_per_day'] = periods_per_day

In [25]:
def gme_local_datetime(row):
    date = row['Date']
    hour = row['Ora']
    periods = row['periods_per_day']

    # normal day: 24 periods
    if periods == 24:
        return date + pd.to_timedelta(hour - 1, unit='h')

    # spring DST: 23 periods
    # Ora 1 = 00:00
    # Ora 2 = 01:00
    # Ora 3 = 03:00
    elif periods == 23:
        if hour >= 3:
            return date + pd.to_timedelta(hour, unit='h')
        else:
            return date + pd.to_timedelta(hour - 1, unit='h')

    # Autumn DST: 25 periods
    # Ora 1 = 00:00
    # Ora 2 = 01:00
    # Ora 3 = 02:00 CEST
    # Ora 4 = 02:00 CET
    # Ora 5 = 03:00 CET
    elif periods == 25:
        if hour <= 2:
            return date + pd.to_timedelta(hour - 1, unit='h')
        elif hour <= 4:
            return date + pd.Timedelta(hours=2)
        else:
            return date + pd.to_timedelta(hour - 2, unit='h')

    else:
        raise ValueError(
            f"Unexpected number of periods ({periods}) for {date}"
        )

In [26]:
load_data['datetime_local_naive'] = load_data.apply(
    gme_local_datetime,
    axis=1
)

In [27]:
ambiguous = (
    (load_data['periods_per_day'] == 25)
    & (load_data['Ora'] == 3)
)

load_data['datetime_local'] = (
    load_data['datetime_local_naive']
    .dt.tz_localize(
        'Europe/Rome',
        ambiguous=ambiguous,
        nonexistent='raise'
    )
)

In [28]:
load_data['datetime_UTC'] = (
    load_data['datetime_local']
    .dt.tz_convert('UTC')
)

In [29]:
load_data.head()

,Data,Ora,Centro Nord,Centro Sud,Nord,Sardegna,Sicilia,Sud,Totale Italia,Calabria,datetime,Date,periods_per_day,datetime_local_naive,datetime_local,datetime_UTC
0,2015-01-01,1,"2.670,851","4.590,186","10.774,037",997.138,"1.930,829","2.347,270","23.310,311",NaN,2015-01-01 00:00:00,2015-01-01,24,2015-01-01 00:00:00,2015-01-01 00:00:00+01:00,2014-12-31 23:00:00+00:00
1,2015-01-01,2,"2.513,713","4.342,739","10.576,993",951.110,"1.821,231","2.213,368","22.419,154",NaN,2015-01-01 01:00:00,2015-01-01,24,2015-01-01 01:00:00,2015-01-01 01:00:00+01:00,2015-01-01 00:00:00+00:00
2,2015-01-01,3,"2.392,789","4.028,419","10.169,339",903.001,"1.769,136","2.031,384","21.294,068",NaN,2015-01-01 02:00:00,2015-01-01,24,2015-01-01 02:00:00,2015-01-01 02:00:00+01:00,2015-01-01 01:00:00+00:00
3,2015-01-01,4,"2.259,362","3.732,403","9.769,460",855.196,"1.673,111","1.869,842","20.159,374",NaN,2015-01-01 03:00:00,2015-01-01,24,2015-01-01 03:00:00,2015-01-01 03:00:00+01:00,2015-01-01 02:00:00+00:00
4,2015-01-01,5,"2.245,990","3.363,157","9.613,212",855.747,"1.536,739","1.761,414","19.376,259",NaN,2015-01-01 04:00:00,2015-01-01,24,2015-01-01 04:00:00,2015-01-01 04:00:00+01:00,2015-01-01 03:00:00+00:00


In [30]:
load_data.loc[
    load_data['Date'] == '2015-10-25',
    ['Date', 'Ora', 'datetime_local', 'datetime_UTC']
].to_string(index=False)

'      Date  Ora            datetime_local              datetime_UTC\n2015-10-25    1 2015-10-25 00:00:00+02:00 2015-10-24 22:00:00+00:00\n2015-10-25    2 2015-10-25 01:00:00+02:00 2015-10-24 23:00:00+00:00\n2015-10-25    3 2015-10-25 02:00:00+02:00 2015-10-25 00:00:00+00:00\n2015-10-25    4 2015-10-25 02:00:00+01:00 2015-10-25 01:00:00+00:00\n2015-10-25    5 2015-10-25 03:00:00+01:00 2015-10-25 02:00:00+00:00\n2015-10-25    6 2015-10-25 04:00:00+01:00 2015-10-25 03:00:00+00:00\n2015-10-25    7 2015-10-25 05:00:00+01:00 2015-10-25 04:00:00+00:00\n2015-10-25    8 2015-10-25 06:00:00+01:00 2015-10-25 05:00:00+00:00\n2015-10-25    9 2015-10-25 07:00:00+01:00 2015-10-25 06:00:00+00:00\n2015-10-25   10 2015-10-25 08:00:00+01:00 2015-10-25 07:00:00+00:00\n2015-10-25   11 2015-10-25 09:00:00+01:00 2015-10-25 08:00:00+00:00\n2015-10-25   12 2015-10-25 10:00:00+01:00 2015-10-25 09:00:00+00:00\n2015-10-25   13 2015-10-25 11:00:00+01:00 2015-10-25 10:00:00+00:00\n2015-10-25   14 2015-10-25 12:00:

In [31]:
load_data['datetime_UTC'].diff().value_counts().head()

datetime_UTC
0 days 01:00:00    100774
Name: count, dtype: int64

In [32]:
load_data.loc[
    load_data['Date'].isin([
        '2015-03-29',
        '2015-10-25',
        '2015-10-26'
    ]),
    ['Date', 'Ora', 'datetime_local', 'datetime_UTC']
].to_string(index=False)

'Empty DataFrame\nColumns: [Date, Ora, datetime_local, datetime_UTC]\nIndex: []'

In [33]:
print(
    load_data['datetime_UTC'].isna().sum()
)

0


In [34]:
load_data = load_data.drop(
    columns=['periods_per_day', 'datetime_local_naive']
)

In [35]:
load_data = load_data[
    ['Date', 'Ora', 'datetime_local', 'datetime_UTC']
    + [
        col for col in load_data.columns
        if col not in [
            'Date',
            'Ora',
            'datetime_local',
            'datetime_UTC'
        ]
    ]
]

In [36]:
load_data[
    load_data['Date'].dt.strftime('%Y-%m-%d') == '2015-03-29'
][['Date', 'Ora', 'datetime_local', 'datetime_UTC']]

,Date,Ora,datetime_local,datetime_UTC
2088,2015-03-29,1,2015-03-29 00:00:00+01:00,2015-03-28 23:00:00+00:00
2089,2015-03-29,2,2015-03-29 01:00:00+01:00,2015-03-29 00:00:00+00:00
2090,2015-03-29,3,2015-03-29 03:00:00+02:00,2015-03-29 01:00:00+00:00
2091,2015-03-29,4,2015-03-29 04:00:00+02:00,2015-03-29 02:00:00+00:00
2092,2015-03-29,5,2015-03-29 05:00:00+02:00,2015-03-29 03:00:00+00:00
2093,2015-03-29,6,2015-03-29 06:00:00+02:00,2015-03-29 04:00:00+00:00
2094,2015-03-29,7,2015-03-29 07:00:00+02:00,2015-03-29 05:00:00+00:00
2095,2015-03-29,8,2015-03-29 08:00:00+02:00,2015-03-29 06:00:00+00:00
2096,2015-03-29,9,2015-03-29 09:00:00+02:00,2015-03-29 07:00:00+00:00
2097,2015-03-29,10,2015-03-29 10:00:00+02:00,2015-03-29 08:00:00+00:00


In [37]:
load_data.loc[
    load_data['Date'].dt.strftime('%Y-%m-%d') == '2015-03-29',
    ['Date', 'Ora', 'datetime_local', 'datetime_UTC']
].to_string(index=False)

'      Date  Ora            datetime_local              datetime_UTC\n2015-03-29    1 2015-03-29 00:00:00+01:00 2015-03-28 23:00:00+00:00\n2015-03-29    2 2015-03-29 01:00:00+01:00 2015-03-29 00:00:00+00:00\n2015-03-29    3 2015-03-29 03:00:00+02:00 2015-03-29 01:00:00+00:00\n2015-03-29    4 2015-03-29 04:00:00+02:00 2015-03-29 02:00:00+00:00\n2015-03-29    5 2015-03-29 05:00:00+02:00 2015-03-29 03:00:00+00:00\n2015-03-29    6 2015-03-29 06:00:00+02:00 2015-03-29 04:00:00+00:00\n2015-03-29    7 2015-03-29 07:00:00+02:00 2015-03-29 05:00:00+00:00\n2015-03-29    8 2015-03-29 08:00:00+02:00 2015-03-29 06:00:00+00:00\n2015-03-29    9 2015-03-29 09:00:00+02:00 2015-03-29 07:00:00+00:00\n2015-03-29   10 2015-03-29 10:00:00+02:00 2015-03-29 08:00:00+00:00\n2015-03-29   11 2015-03-29 11:00:00+02:00 2015-03-29 09:00:00+00:00\n2015-03-29   12 2015-03-29 12:00:00+02:00 2015-03-29 10:00:00+00:00\n2015-03-29   13 2015-03-29 13:00:00+02:00 2015-03-29 11:00:00+00:00\n2015-03-29   14 2015-03-29 14:00:

In [39]:
load_data.columns

Index(['Date', 'Ora', 'datetime_local', 'datetime_UTC', 'Data', 'Centro Nord',
       'Centro Sud', 'Nord', 'Sardegna', 'Sicilia', 'Sud', 'Totale Italia',
       'Calabria', 'datetime'],
      dtype='str')

In [40]:
load_data = load_data.drop(
    columns=['Data', 'datetime']
)

In [41]:
# final cleaning check
load_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100775 entries, 0 to 100774
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype                      
---  ------          --------------   -----                      
 0   Date            100775 non-null  datetime64[us]             
 1   Ora             100775 non-null  int64                      
 2   datetime_local  100775 non-null  datetime64[us, Europe/Rome]
 3   datetime_UTC    100775 non-null  datetime64[us, UTC]        
 4   Centro Nord     100775 non-null  str                        
 5   Centro Sud      100775 non-null  str                        
 6   Nord            100775 non-null  str                        
 7   Sardegna        100775 non-null  str                        
 8   Sicilia         100775 non-null  str                        
 9   Sud             100775 non-null  str                        
 10  Totale Italia   100775 non-null  str                        
 11  Calabria        48167 non-null   obje

In [46]:
loads = load_data.columns[4:]

for load in loads:
    load_data[load] = (
        load_data[load]
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .astype(float)
    )

In [48]:
load_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100775 entries, 0 to 100774
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype                      
---  ------          --------------   -----                      
 0   Date            100775 non-null  datetime64[us]             
 1   Ora             100775 non-null  int64                      
 2   datetime_local  100775 non-null  datetime64[us, Europe/Rome]
 3   datetime_UTC    100775 non-null  datetime64[us, UTC]        
 4   Centro Nord     100775 non-null  float64                    
 5   Centro Sud      100775 non-null  float64                    
 6   Nord            100775 non-null  float64                    
 7   Sardegna        100775 non-null  float64                    
 8   Sicilia         100775 non-null  float64                    
 9   Sud             100775 non-null  float64                    
 10  Totale Italia   100775 non-null  float64                    
 11  Calabria        43824 non-null   floa

In [50]:
load_data.columns.tolist()

['Date',
 'Ora',
 'datetime_local',
 'datetime_UTC',
 'Centro Nord',
 'Centro Sud',
 'Nord',
 'Sardegna',
 'Sicilia',
 'Sud',
 'Totale Italia',
 'Calabria']

In [51]:
from pathlib import Path

output_path = Path('../dataset/clean/load')

# create folder if non-existent
output_path.mkdir(parents=True, exist_ok=True)

# save in CSV format
load_data.to_csv(output_path / 'load-clean.csv', index=False)